In [4]:
import os
import sys
import argparse
import glob
import json
import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from pathlib import Path
from scipy import ndimage

In [5]:
import os, sys

# Fix for Jupyter: use current working directory instead of __file__
MONODEPTH2_DIR = os.path.join(os.getcwd(), "monodepth2")

def setup_monodepth2():
    if not os.path.isdir(MONODEPTH2_DIR):
        print("[setup] Cloning MonoDepth2 …")
        ret = os.system(
            "git clone https://github.com/nianticlabs/monodepth2.git "
            + MONODEPTH2_DIR
        )
        if ret != 0:
            sys.exit("ERROR: Could not clone MonoDepth2. Check network access.")
    if MONODEPTH2_DIR not in sys.path:
        sys.path.insert(0, MONODEPTH2_DIR)

setup_monodepth2()

In [6]:
import torch
import torchvision.transforms as transforms

# MonoDepth2 ships its own networks module
sys.path.insert(0, MONODEPTH2_DIR)
import networks
from layers import disp_to_depth


# ──────────────────────────────────────────────
# 3.  Model loader
# ──────────────────────────────────────────────
MODEL_URLS = {
    "mono_640x192":         "https://storage.googleapis.com/niantic-lon-static/research/monodepth2/mono_640x192.zip",
    "stereo_640x192":       "https://storage.googleapis.com/niantic-lon-static/research/monodepth2/stereo_640x192.zip",
    "mono+stereo_640x192":  "https://storage.googleapis.com/niantic-lon-static/research/monodepth2/mono%2Bstereo_640x192.zip",
    "mono_1024x320":        "https://storage.googleapis.com/niantic-lon-static/research/monodepth2/mono_1024x320.zip",
    "mono+stereo_1024x320": "https://storage.googleapis.com/niantic-lon-static/research/monodepth2/mono%2Bstereo_1024x320.zip",
}

def download_model(model_name: str, models_root: str = "models") -> str:
    """Download & unzip pretrained weights if not present."""
    model_path = os.path.join(models_root, model_name)
    if os.path.isdir(model_path) and os.listdir(model_path):
        print(f"[model] Found cached weights at {model_path}")
        return model_path

    os.makedirs(models_root, exist_ok=True)
    url = MODEL_URLS[model_name]
    zip_path = os.path.join(models_root, f"{model_name}.zip")
    print(f"[model] Downloading {model_name} …")
    
    # Use Python's urllib instead of wget (works on Mac + Linux)
    import urllib.request
    urllib.request.urlretrieve(url, zip_path)
    
    os.system(f'unzip -q "{zip_path}" -d "{models_root}"')
    os.remove(zip_path)
    return model_path

In [7]:
def load_model(model_name: str, models_root: str = "models"):
    """Load encoder + depth decoder, return (encoder, depth_decoder, feed_wh)."""
    model_path = download_model(model_name, models_root)

    # Infer feed resolution from name
    if "1024x320" in model_name:
        feed_width, feed_height = 1024, 320
    else:
        feed_width, feed_height = 640, 192

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if device is "CPU":
        device = torch.device("mps" if torch.mps.is_available() else "cpu")

    print(f"[model] Using device: {device}")

    # Encoder
    encoder = networks.ResnetEncoder(18, False)
    encoder_path = os.path.join(model_path, "encoder.pth")
    loaded = torch.load(encoder_path, map_location=device)
    # strip keys not in model
    filtered = {k: v for k, v in loaded.items() if k in encoder.state_dict()}
    encoder.load_state_dict(filtered)
    encoder.to(device).eval()

    # Depth decoder
    depth_decoder = networks.DepthDecoder(
        num_ch_enc=encoder.num_ch_enc, scales=range(4)
    )
    depth_path = os.path.join(model_path, "depth.pth")
    depth_decoder.load_state_dict(
        torch.load(depth_path, map_location=device)
    )
    depth_decoder.to(device).eval()

    return encoder, depth_decoder, feed_width, feed_height, device

<>:13: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:13: SyntaxWarning: "is" with a literal. Did you mean "=="?
/var/folders/f2/t74vhry566j4hf8364yxtd_40000gn/T/ipykernel_7896/2187530087.py:13: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if device is "CPU":


In [8]:
DEPTH_MIN_METRES = 0.1     # MonoDepth2 min-max scaling constants
DEPTH_MAX_METRES = 100.0
# def infer_depth(image_path, encoder, depth_decoder,
#                 feed_width, feed_height, device):
#     img = Image.open(image_path).convert("RGB")

#     img_resized = img.resize((feed_width, feed_height), Image.LANCZOS)
#     img_tensor = transforms.ToTensor()(img_resized).unsqueeze(0).to(device)

#     with torch.no_grad():
#         features = encoder(img_tensor)
#         outputs  = depth_decoder(features)

#     disp = outputs[("disp", 0)]  # shape: (1, 1, 192, 640)

#     disp_np = disp.squeeze().cpu().numpy()  # (192, 640)

#     # Invert disparity → depth (far = large value)
#     depth_map = 1.0 / (disp_np + 1e-8)

#     # Normalize to [0, 1]
#     depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-8)

#     return depth_map, (feed_width, feed_height)
DEPTH_MIN_METRES = 0.1
DEPTH_MAX_METRES = 100.0

def infer_depth(image_path, encoder, depth_decoder,
                feed_width, feed_height, device):
    img = Image.open(image_path).convert("RGB")

    img_resized = img.resize((feed_width, feed_height), Image.LANCZOS)
    img_tensor = transforms.ToTensor()(img_resized).unsqueeze(0).to(device)

    with torch.no_grad():
        features = encoder(img_tensor)
        outputs  = depth_decoder(features)

    disp = outputs[("disp", 0)]  # (1, 1, 192, 640)

    # Convert disparity → metric depth in metres
    _, depth = disp_to_depth(disp, DEPTH_MIN_METRES, DEPTH_MAX_METRES)

    depth_np = depth.squeeze().cpu().numpy()  # (192, 640), values in metres

    return depth_np, (feed_width, feed_height)

In [9]:
CMAP = "plasma"   # vivid, perceptually uniform; alternatives: inferno, magma, viridis

def save_heatmap(depth_map, out_path, orig_image_path=None):
    vmin = np.percentile(depth_map, 1)
    vmax = np.percentile(depth_map, 99)
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    cmap = "plasma"  # purple=close, yellow=far — matches depth convention

    fig, axes = plt.subplots(1, 2 if orig_image_path else 1,
                             figsize=(20 if orig_image_path else 10, 6),
                             tight_layout=True)
    if orig_image_path:
        ax_img, ax_dep = axes
        ax_img.imshow(Image.open(orig_image_path).convert("RGB"))
        ax_img.set_title("Original Frame", fontsize=12)
        ax_img.axis("off")
    else:
        ax_dep = axes

    ax_dep.imshow(depth_map, cmap=cmap, norm=norm)
    ax_dep.set_title("MonoDepth2 – Depth Heatmap", fontsize=12)
    ax_dep.axis("off")

    cbar = fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap),
        ax=ax_dep, fraction=0.03, pad=0.02
    )
    cbar.set_label("Depth (m)", fontsize=10)

    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)


# ──────────────────────────────────────────────
# 6.  Per-frame statistics (matching your schema)
# ──────────────────────────────────────────────
VALID_DEPTH_MIN = 0.5    # metres – below this = likely artefact / sky
VALID_DEPTH_MAX = 80.0   # metres – above this = unreliable mono estimate

def compute_frame_stats(depth_map: np.ndarray, frame_index: int) -> dict:
    """
    Returns a dict matching the requested schema:
        frame_index, valid_pixels, percent_valid,
        connected_objects, unique_depth_values
    """
    total_pixels = depth_map.size
    valid_mask   = (depth_map >= VALID_DEPTH_MIN) & (depth_map <= VALID_DEPTH_MAX)
    valid_pixels = int(valid_mask.sum())
    percent_valid = round(valid_pixels / total_pixels * 100, 2)

    # Connected components on the valid mask → proxy for "objects"
    labeled, connected_objects = ndimage.label(valid_mask)

    # Unique depth values (rounded to cm for counting)
    depth_cm = (depth_map[valid_mask] * 100).astype(np.int32)
    unique_depth_values = int(np.unique(depth_cm).size)

    return {
        "frame_index":        frame_index,
        "valid_pixels":       np.int64(valid_pixels),
        "percent_valid":      np.float64(percent_valid),
        "connected_objects":  connected_objects,
        "unique_depth_values": unique_depth_values,
    }


# ──────────────────────────────────────────────
# 7.  Per-pixel depth record  (xi, yi → depth)
# ──────────────────────────────────────────────
def build_pixel_depth_map(depth_map: np.ndarray, frame_index: int) -> dict:
    """
    Returns:
        {
          "frame_index": int,
          "depth_array": np.ndarray shape (H, W) float32   ← full grid
        }
    Store as a dict keyed by frame_index in the master list.
    For explicit (xi, yi, depth) triples you can call pixel_triplets().
    """
    return {
        "frame_index": frame_index,
        "depth_array": depth_map.astype(np.float32),
    }


def pixel_triplets(depth_map: np.ndarray):
    """Return (N,3) array of [x, y, depth] for every pixel."""
    H, W = depth_map.shape
    ys, xs = np.mgrid[0:H, 0:W]
    return np.stack([xs.ravel(), ys.ravel(), depth_map.ravel()], axis=1).astype(np.float32)



In [10]:

def run_pipeline(image_dir: str, output_dir: str, model_name: str,
                 models_root: str = "models", save_side_by_side: bool = True,
                 save_pixel_triplets: bool = False):

    # ── output folders
    heatmap_dir = os.path.join(output_dir, "heatmaps")
    os.makedirs(heatmap_dir, exist_ok=True)

    # ── collect & sort frames
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")
    frames = []
    for ext in exts:
        frames.extend(glob.glob(os.path.join(image_dir, ext)))
    frames = sorted(frames)
    if not frames:
        sys.exit(f"ERROR: No images found in {image_dir}")
    print(f"[pipeline] Found {len(frames)} frames.")

    # ── load model
    encoder, depth_decoder, feed_w, feed_h, device = load_model(
        model_name, models_root
    )

    # ── accumulators
    stats_list      = []   # list of per-frame stat dicts (your schema)
    depth_maps_list = []   # list of {"frame_index": int, "depth_array": ndarray}

    for idx, frame_path in enumerate(frames):
        print(f"  [{idx+1:>3}/{len(frames)}] {os.path.basename(frame_path)}", end=" … ")

        depth_map, orig_size = infer_depth(
            frame_path, encoder, depth_decoder, feed_w, feed_h, device
        )

        # -- heatmap
        heatmap_name = f"depth_{idx:04d}.png"
        save_heatmap(
            depth_map,
            os.path.join(heatmap_dir, heatmap_name),
            orig_image_path=frame_path if save_side_by_side else None,
        )

        # -- stats
        stats = compute_frame_stats(depth_map, frame_index=idx)
        stats_list.append(stats)

        # -- pixel depth map
        depth_maps_list.append(build_pixel_depth_map(depth_map, frame_index=idx))

        print(
            f"valid={stats['valid_pixels']:,}  "
            f"({stats['percent_valid']}%)  "
            f"objs={stats['connected_objects']}  "
            f"udv={stats['unique_depth_values']}"
        )

    # ── save stats as JSON (human-readable)
    stats_json_path = os.path.join(output_dir, "frame_stats.json")
    json_safe = []
    for s in stats_list:
        json_safe.append({
            "frame_index":        int(s["frame_index"]),
            "valid_pixels":       int(s["valid_pixels"]),
            "percent_valid":      float(s["percent_valid"]),
            "connected_objects":  int(s["connected_objects"]),
            "unique_depth_values": int(s["unique_depth_values"]),
        })
    with open(stats_json_path, "w") as f:
        json.dump(json_safe, f, indent=2)
    print(f"\n[output] Stats JSON  → {stats_json_path}")

    # ── save full depth arrays as numpy / pickle
    depth_npy_path = os.path.join(output_dir, "all_depth_maps.pkl")
    with open(depth_npy_path, "wb") as f:
        pickle.dump(depth_maps_list, f)
    print(f"[output] Depth maps  → {depth_npy_path}  ({len(depth_maps_list)} frames)")

    # ── optionally also save (xi, yi, depth) triplet arrays per frame
    if save_pixel_triplets:
        triplets_dir = os.path.join(output_dir, "pixel_triplets")
        os.makedirs(triplets_dir, exist_ok=True)
        for entry in depth_maps_list:
            t = pixel_triplets(entry["depth_array"])
            np.save(
                os.path.join(triplets_dir, f"triplets_{entry['frame_index']:04d}.npy"),
                t,
            )
        print(f"[output] Pixel triplets → {triplets_dir}/")

    print("\n✅ Pipeline complete.")
    print(f"   Heatmaps  : {heatmap_dir}/")
    print(f"   Stats     : {stats_json_path}")
    print(f"   Depth maps: {depth_npy_path}")

    return stats_list, depth_maps_list


# ──────────────────────────────────────────────
# 9.  Loading helpers (use after pipeline)
# ──────────────────────────────────────────────
def load_stats(output_dir: str):
    """Load the per-frame stats list back from JSON."""
    path = os.path.join(output_dir, "frame_stats.json")
    with open(path) as f:
        raw = json.load(f)
    result = []
    for s in raw:
        result.append({
            "frame_index":        s["frame_index"],
            "valid_pixels":       np.int64(s["valid_pixels"]),
            "percent_valid":      np.float64(s["percent_valid"]),
            "connected_objects":  s["connected_objects"],
            "unique_depth_values": s["unique_depth_values"],
        })
    return result


def load_depth_maps(output_dir: str):
    """Load all depth arrays. Returns list of dicts with 'frame_index' and 'depth_array'."""
    path = os.path.join(output_dir, "all_depth_maps.pkl")
    with open(path, "rb") as f:
        return pickle.load(f)


def get_pixel_depth(depth_maps_list, frame_index: int, xi: int, yi: int) -> float:
    """Quick lookup: depth at pixel (xi, yi) for a given frame."""
    entry = next(e for e in depth_maps_list if e["frame_index"] == frame_index)
    return float(entry["depth_array"][yi, xi])


# ──────────────────────────────────────────────
# 10. CLI entry-point
# ──────────────────────────────────────────────
# ── Config ──────────────────────────────────────
IMAGE_DIR        = "images"    # relative to where the notebook lives
OUTPUT_DIR       = "output"    # the output/ folder already exists in your tree
MODEL_NAME       = "mono+stereo_640x192"
MODELS_ROOT      = "models"    # will be created here automatically
SAVE_SIDE_BY_SIDE = True
SAVE_TRIPLETS     = False

# ── Run ─────────────────────────────────────────
stats_list, depth_maps_list = run_pipeline(
    image_dir           = IMAGE_DIR,
    output_dir          = OUTPUT_DIR,
    model_name          = MODEL_NAME,
    models_root         = MODELS_ROOT,
    save_side_by_side   = SAVE_SIDE_BY_SIDE,
    save_pixel_triplets = SAVE_TRIPLETS,
)

# ── Preview stats ────────────────────────────────
for s in stats_list[:5]:
    print(s)

/Users/devrathod/miniconda3/envs/tf_env/lib/python3.11/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/Users/devrathod/miniconda3/envs/tf_env/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


[pipeline] Found 199 frames.
[model] Found cached weights at models/mono+stereo_640x192
[model] Using device: cpu


/var/folders/f2/t74vhry566j4hf8364yxtd_40000gn/T/ipykernel_7896/2187530087.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded = torch.load(encoder_path, map_locatio

  [  1/199] frame_00000.png … valid=122,880  (100.0%)  objs=1  udv=2159
  [  2/199] frame_00001.png … valid=122,880  (100.0%)  objs=1  udv=2053
  [  3/199] frame_00002.png … valid=122,880  (100.0%)  objs=1  udv=2098
  [  4/199] frame_00003.png … valid=122,880  (100.0%)  objs=1  udv=2269
  [  5/199] frame_00004.png … valid=122,880  (100.0%)  objs=1  udv=2451
  [  6/199] frame_00005.png … valid=122,880  (100.0%)  objs=1  udv=2133
  [  7/199] frame_00006.png … valid=122,880  (100.0%)  objs=1  udv=2202
  [  8/199] frame_00007.png … valid=122,880  (100.0%)  objs=1  udv=2290
  [  9/199] frame_00008.png … valid=122,880  (100.0%)  objs=1  udv=2383
  [ 10/199] frame_00009.png … valid=122,880  (100.0%)  objs=1  udv=2378
  [ 11/199] frame_00010.png … valid=122,880  (100.0%)  objs=1  udv=2176
  [ 12/199] frame_00011.png … valid=122,880  (100.0%)  objs=1  udv=1948
  [ 13/199] frame_00012.png … valid=122,880  (100.0%)  objs=1  udv=2046
  [ 14/199] frame_00013.png … valid=122,880  (100.0%)  objs=1  u

In [12]:
import pickle
import numpy as np
from scipy import ndimage
from pathlib import Path
import csv

def analyze_monodepth2_stack(pkl_file, csv_file=None, quantize_depth=False, round_digits=2):
    with open(pkl_file, "rb") as f:
        depth_maps_list = pickle.load(f)

    N = len(depth_maps_list)
    H, W = depth_maps_list[0]["depth_array"].shape
    total_pixels = H * W

    print(f"Loaded: {pkl_file}")
    print(f"Shape: ({N}, {H}, {W})  |  Pixels per frame: {total_pixels}\n")

    results = []
    for entry in depth_maps_list:
        i     = entry["frame_index"]
        frame = entry["depth_array"]
        frame_q = np.round(frame, round_digits) if quantize_depth else frame

        valid_pixels  = np.count_nonzero(frame)
        percent_valid = round((valid_pixels / total_pixels) * 100, 2)
        _, num_objects = ndimage.label(frame > 0)
        unique_depths  = len(np.unique(frame_q))

        results.append({
            "frame_index":         i,
            "valid_pixels":        valid_pixels,
            "percent_valid":       percent_valid,
            "connected_objects":   num_objects,
            "unique_depth_values": unique_depths,
        })

    for r in results:
        print(
            f"Frame {r['frame_index']:>3} | "
            f"valid={r['valid_pixels']:,} | "
            f"{r['percent_valid']}% | "
            f"objects={r['connected_objects']} | "
            f"unique_depths={r['unique_depth_values']}"
        )

    if csv_file:
        with open(csv_file, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=results[0].keys())
            writer.writeheader()
            writer.writerows(results)
        print(f"\nCSV saved → {csv_file}")

    # Save stacked (199, H, W) .npy — same format as your ground truth
    stack = np.stack([e["depth_array"] for e in depth_maps_list], axis=0)
    npy_path = Path(pkl_file).parent / "monodepth2_depths.npy"
    np.save(npy_path, stack)
    print(f"Stacked .npy saved → {npy_path}  shape={stack.shape}")

    return results, stack


# ── Run ──────────────────────────────────────────
results, stack = analyze_monodepth2_stack(
    pkl_file       = "output/all_depth_maps.pkl",
    csv_file       = "output/monodepth2_analysis.csv",
    quantize_depth = True,
    round_digits   = 2,
)

print("\nFinal stack shape:", stack.shape)  # (199, H, W)

Loaded: output/all_depth_maps.pkl
Shape: (199, 192, 640)  |  Pixels per frame: 122880

Frame   0 | valid=122,880 | 100.0% | objects=1 | unique_depths=2176
Frame   1 | valid=122,880 | 100.0% | objects=1 | unique_depths=2040
Frame   2 | valid=122,880 | 100.0% | objects=1 | unique_depths=2090
Frame   3 | valid=122,880 | 100.0% | objects=1 | unique_depths=2279
Frame   4 | valid=122,880 | 100.0% | objects=1 | unique_depths=2446
Frame   5 | valid=122,880 | 100.0% | objects=1 | unique_depths=2124
Frame   6 | valid=122,880 | 100.0% | objects=1 | unique_depths=2194
Frame   7 | valid=122,880 | 100.0% | objects=1 | unique_depths=2303
Frame   8 | valid=122,880 | 100.0% | objects=1 | unique_depths=2377
Frame   9 | valid=122,880 | 100.0% | objects=1 | unique_depths=2371
Frame  10 | valid=122,880 | 100.0% | objects=1 | unique_depths=2173
Frame  11 | valid=122,880 | 100.0% | objects=1 | unique_depths=1939
Frame  12 | valid=122,880 | 100.0% | objects=1 | unique_depths=2042
Frame  13 | valid=122,880 | 1